In [39]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import ( col, when, trim, lit, regexp_replace, split, explode, lower, array_contains, transform, isnan, count, round)
from pyspark.sql.types import IntegerType
from delta import *

warehouse_location = 'hdfs://hdfs-nn:9000/projeto/silver'

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("ETL-Netflix-Titles-Silver") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada com sucesso e Delta Lake configurado.")

SparkSession iniciada com sucesso e Delta Lake configurado.


In [40]:
hdfs_path = "hdfs://hdfs-nn:9000/projeto/bronze/netflix_titles.csv"

In [41]:
df_bronze = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True,
    quote='\"',
    escape='\"',
    multiLine=True
)

In [42]:
print(f"Total de registos lidos da camada Bronze: {df_bronze.count()}")
df_bronze.printSchema()
df_bronze.toPandas()

Total de registos lidos da camada Bronze: 5850
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: double (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)



,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,None,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,None,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5845,tm1014599,Fine Wine,MOVIE,A beautiful love story that can happen between...,2021,None,100,"['romance', 'drama']",['NG'],NaN,tt13857480,6.8,45.0,1.466,NaN
5846,tm898842,C/O Kaadhal,MOVIE,A heart warming film that explores the concept...,2021,None,134,['drama'],[],NaN,tt11803618,7.7,348.0,NaN,NaN
5847,tm1059008,Lokillo,MOVIE,A controversial TV host and comedian who has b...,2021,None,90,['comedy'],['CO'],NaN,tt14585902,3.8,68.0,26.005,6.300
5848,tm1035612,Dad Stop Embarrassing Me - The Afterparty,MOVIE,"Jamie Foxx, David Alan Grier and more from the...",2021,PG-13,37,[],['US'],NaN,None,NaN,NaN,1.296,10.000


In [43]:
from pyspark.sql.functions import col, count

print(f"Total de registos ANTES da limpeza dos ids: {df_bronze.count()}")
print("IDs duplicados (contagem > 1):")

# Agrupa por 'id', conta as ocorrências e filtra apenas os que aparecem mais de uma vez
df_bronze.groupBy("id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

Total de registos ANTES da limpeza dos ids: 5850
IDs duplicados (contagem > 1):
+---+-----+
| id|count|
+---+-----+
+---+-----+



In [6]:
# O método .dropDuplicates() remove linhas inteiras baseadas na(s) coluna(s) fornecida(s)
# Ele mantém a *primeira* ocorrência que encontra.
df_id = df_bronze.dropDuplicates(['id'])

print("Registos duplicados removidos.")

Registos duplicados removidos.


In [7]:
from pyspark.sql.functions import col, count

print("Verificar se ainda existem IDs duplicados...")

duplicados_apos_limpeza = df_id.groupBy("id") \
    .count() \
    .filter(col("count") > 1)

num_duplicados = duplicados_apos_limpeza.count()

if num_duplicados == 0:
    print("✅ SUCESSO: Não foram encontrados mais IDs duplicados.")
else:
    print(f"❌ FALHA: Ainda existem {num_duplicados} IDs duplicados.")
    duplicados_apos_limpeza.show()

print("---")
print(f"Total de registos DEPOIS da limpeza: {df_id.count()}")

Verificar se ainda existem IDs duplicados...
✅ SUCESSO: Não foram encontrados mais IDs duplicados.
---
Total de registos DEPOIS da limpeza: 5850


In [8]:
from pyspark.sql.functions import col, trim, count

print(f"Total de registos no DataFrame 'df_id': {df_id.count()}")

# Procura por linhas onde 'title' é nulo ou uma string vazia 
titulos_vazios = df_id.filter(
    (col("title").isNull()) | (trim(col("title")) == "")
)

titulos_vazios.select("id", "title").show()
print(f"Total de títulos nulos/vazios encontrados: {titulos_vazios.count()}")

Total de registos no DataFrame 'df_id': 5850
+---------+-----+
|       id|title|
+---------+-----+
|tm1063792| null|
+---------+-----+

Total de títulos nulos/vazios encontrados: 1


In [9]:
print(f"Registos ANTES da limpeza 'title': {df_id.count()}")

df_titles = df_id.filter(
    (col("title").isNotNull()) & (trim(col("title")) != "")
)

print(f"Registos DEPOIS da limpeza 'title': {df_titles.count()}")

Registos ANTES da limpeza 'title': 5850
Registos DEPOIS da limpeza 'title': 5849


In [10]:
("Verificar se ainda existem títulos nulos/vazios...")

titulos_vazios_depois = df_titles.filter(
    (col("title").isNull()) | (trim(col("title")) == "")
)

num_vazios = titulos_vazios_depois.count()

if num_vazios == 0:
    print("✅ SUCESSO: Não foram encontrados mais títulos nulos/vazios.")
else:
    print(f"❌ FALHA: Ainda existem {num_vazios} títulos nulos/vazios.")
    titulos_vazios_depois.show()

✅ SUCESSO: Não foram encontrados mais títulos nulos/vazios.


In [11]:
from pyspark.sql.functions import col, trim, count

print(f"Total de registos no DataFrame 'df_titles': {df_titles.count()}")

# Procura por linhas onde 'description' é nulo ou uma string vazia 
descricoes_vazias = df_titles.filter(
    (col("description").isNull()) | (trim(col("description")) == "")
)

descricoes_vazias.select("id", "title", "description").show()
print(f"Total de descrições nulas/vazias encontradas: {descricoes_vazias.count()}")

Total de registos no DataFrame 'df_titles': 5849
+---------+--------------------+-----------+
|       id|               title|description|
+---------+--------------------+-----------+
|tm1172010|   The Lockdown Plan|       null|
|tm1237253|My Little Pony: A...|       null|
| tm407349|  The Birth Reborn 2|       null|
| tm681614|  Grandmother's Farm|       null|
| tm840152|Amsterdam to Anat...|       null|
| tm855232|Luccas Neto in: T...|       null|
| tm896976|True: Rainbow Rescue|       null|
| tm902993|          Mama Drama|       null|
| ts106612|Chicken Soup for ...|       null|
| ts223062|   ปริศนาล่าข้ามเวลา|       null|
| ts255867|     뽀로로 동화나라|       null|
| ts268283|        RIDE ON TIME|       null|
| ts271005|            幸福料理|       null|
| ts304136|       마법버스 타요|       null|
| ts312044|Beyblade Burst Surge|       null|
|  ts82375|Beyblade Burst Turbo|       null|
|  ts82770|  Camarón Revolution|       null|
+---------+--------------------+-----------+

Total de descrições nu

In [12]:
from pyspark.sql.functions import lit

df_description = df_titles.withColumn("description",
    when(
        (col("description").isNull()) | (trim(col("description")) == ""),
        lit("NA")  # Coloca a string literal "NA"
    ).otherwise(col("description")) # Mantém o valor original se não for nulo/vazio
)

In [13]:
descricoes_vazias_depois = df_description.filter(
    (col("description").isNull()) | (trim(col("description")) == "")
)

num_vazias = descricoes_vazias_depois.count()

if num_vazias == 0:
    print("✅ SUCESSO: Não foram encontradas mais descrições nulas/vazias.")
else:
    print(f"❌ FALHA: Ainda existem {num_vazias} descrições nulas/vazias.")
    descricoes_vazias_depois.show()

descricoes_na = df_description.filter(
    col("description") == "NA"
)

print(f"\nTotal de registos com description = 'NA': {descricoes_na.count()}")
descricoes_na.select("id", "title", "description").show(5)

✅ SUCESSO: Não foram encontradas mais descrições nulas/vazias.

Total de registos com description = 'NA': 17
+---------+--------------------+-----------+
|       id|               title|description|
+---------+--------------------+-----------+
|tm1172010|   The Lockdown Plan|         NA|
|tm1237253|My Little Pony: A...|         NA|
| tm407349|  The Birth Reborn 2|         NA|
| tm681614|  Grandmother's Farm|         NA|
| tm840152|Amsterdam to Anat...|         NA|
+---------+--------------------+-----------+
only showing top 5 rows



In [14]:
from pyspark.sql.functions import col, trim, count

print(f"Total de registos no DataFrame 'df_description': {df_description.count()}")

# Procura por linhas onde 'age_certification' é nulo ou uma string vazia
age_vazios = df_description.filter(
    (col("age_certification").isNull()) | (trim(col("age_certification")) == "")
)

age_vazios.select("id", "title", "age_certification").show(5)
print(f"Total de 'age_certification' nulos/vazios encontrados: {age_vazios.count()}")

Total de registos no DataFrame 'df_description': 5849
+---------+--------------------+-----------------+
|       id|               title|age_certification|
+---------+--------------------+-----------------+
|tm1000166|Wave of Cinema: S...|             null|
|tm1000185|        Squared Love|             null|
| tm100027| Alibaba Aur 40 Chor|             null|
|tm1000296|New Gods: Nezha R...|             null|
|tm1000551|      Namaste Wahala|             null|
+---------+--------------------+-----------------+
only showing top 5 rows

Total de 'age_certification' nulos/vazios encontrados: 2618


In [15]:
from pyspark.sql.functions import lit
df_age_certification = df_description.withColumn("age_certification",
    when(
        (col("age_certification").isNull()) | (trim(col("age_certification")) == ""),
        lit("NA")  # Coloca a string literal "NA"
    ).otherwise(col("age_certification")) # Mantém o valor original
)

In [16]:
age_vazios_depois = df_age_certification.filter(
    (col("age_certification").isNull()) | (trim(col("age_certification")) == "")
)

num_vazias = age_vazios_depois.count()

if num_vazias == 0:
    print("✅ SUCESSO: Não foram encontrados mais 'age_certification' nulos/vazios.")
else:
    print(f"❌ FALHA: Ainda existem {num_vazias} 'age_certification' nulos/vazios.")
    age_vazios_depois.show()

age_na = df_age_certification.filter(
    col("age_certification") == "NA"
)

print(f"\nTotal de registos com age_certification = 'NA': {age_na.count()}")
age_na.select("id", "title", "age_certification").show(5)

✅ SUCESSO: Não foram encontrados mais 'age_certification' nulos/vazios.

Total de registos com age_certification = 'NA': 2618
+---------+--------------------+-----------------+
|       id|               title|age_certification|
+---------+--------------------+-----------------+
|tm1000166|Wave of Cinema: S...|               NA|
|tm1000185|        Squared Love|               NA|
| tm100027| Alibaba Aur 40 Chor|               NA|
|tm1000296|New Gods: Nezha R...|               NA|
|tm1000551|      Namaste Wahala|               NA|
+---------+--------------------+-----------------+
only showing top 5 rows



In [17]:
# Mostra o schema, focado na coluna 'seasons'
df_age_certification.printSchema()
# Filtra os nulos para vermos apenas valores reais
df_age_certification.select("seasons").filter(col("seasons").isNotNull()).show(5)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: double (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

+-------+
|seasons|
+-------+
|    3.0|
|    1.0|
|    1.0|
|    1.0|
|    1.0|
+-------+
only showing top 5 rows



In [18]:
from pyspark.sql.types import IntegerType
# Se a coluna tiver valores nulos, o cast mantém os nulos
df_seasons = df_age_certification.withColumn(
    "seasons",
    col("seasons").cast(IntegerType())
)

In [19]:
# Mostra o schema, 'seasons' deve agora ser 'integer'
df_seasons.printSchema()
# Filtra os nulos para vermos apenas valores reais
df_seasons.select("seasons").filter(col("seasons").isNotNull()).show(5)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

+-------+
|seasons|
+-------+
|      3|
|      1|
|      1|
|      1|
|      1|
+-------+
only showing top 5 rows



In [20]:
from pyspark.sql.functions import col, trim, count

print(f"Total de registos no DataFrame 'df_seasons': {df_seasons.count()}")

# Procura por linhas onde 'imdb_id' é nulo ou uma string vazia (após trim)
imdb_vazios = df_seasons.filter(
    (col("imdb_id").isNull()) | (trim(col("imdb_id")) == "")
)

imdb_vazios.select("id", "title", "imdb_id").show(5)
print(f"\nTotal de 'imdb_id' nulos/vazios encontrados: {imdb_vazios.count()}")

Total de registos no DataFrame 'df_seasons': 5849
+---------+--------------------+-------+
|       id|               title|imdb_id|
+---------+--------------------+-------+
|tm1000166|Wave of Cinema: S...|   null|
|tm1019084|Ginny & Georgia -...|   null|
| tm101914|         The Fighter|   null|
|tm1019822|           Connected|   null|
|tm1020326|    Deadly Illusions|   null|
+---------+--------------------+-------+
only showing top 5 rows


Total de 'imdb_id' nulos/vazios encontrados: 403


In [21]:
from pyspark.sql.functions import lit

df_imdb_id = df_seasons.withColumn("imdb_id",
    when(
        (col("imdb_id").isNull()) | (trim(col("imdb_id")) == ""),
        lit("NA")  # Coloca a string literal "NA"
    ).otherwise(col("imdb_id")) # Mantém o valor original
)

In [22]:
imdb_vazios_depois = df_imdb_id.filter(
    (col("imdb_id").isNull()) | (trim(col("imdb_id")) == "")
)

num_vazias = imdb_vazios_depois.count()

if num_vazias == 0:
    print("✅ SUCESSO: Não foram encontrados mais 'imdb_id' nulos/vazios.")
else:
    print(f"❌ FALHA: Ainda existem {num_vazias} 'imdb_id' nulos/vazios.")
    imdb_vazios_depois.show()

imdb_na = df_imdb_id.filter(
    col("imdb_id") == "NA"
)

print(f"\nTotal de registos com imdb_id = 'NA': {imdb_na.count()}")
imdb_na.select("id", "title", "imdb_id").show(5)

✅ SUCESSO: Não foram encontrados mais 'imdb_id' nulos/vazios.

Total de registos com imdb_id = 'NA': 403
+---------+--------------------+-------+
|       id|               title|imdb_id|
+---------+--------------------+-------+
|tm1000166|Wave of Cinema: S...|     NA|
|tm1019084|Ginny & Georgia -...|     NA|
| tm101914|         The Fighter|     NA|
|tm1019822|           Connected|     NA|
|tm1020326|    Deadly Illusions|     NA|
+---------+--------------------+-------+
only showing top 5 rows



In [23]:
# Mostra o schema, focado na coluna 'imdb_votes'
df_imdb_id.printSchema()
df_imdb_id.select("imdb_votes").filter(col("imdb_votes").isNotNull()).show(5)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

+----------+
|imdb_votes|
+----------+
|    1077.0|
|    4146.0|
|   62464.0|
|    3158.0|
|     565.0|
+----------+
only showing top 5 rows



In [24]:
from pyspark.sql.types import IntegerType

df_imdb_votes = df_imdb_id.withColumn(
    "imdb_votes",
    col("imdb_votes").cast(IntegerType())
)

In [25]:
df_imdb_votes.printSchema()
df_imdb_votes.select("imdb_votes").filter(col("imdb_votes").isNotNull()).show(5)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

+----------+
|imdb_votes|
+----------+
|      1077|
|      4146|
|     62464|
|      3158|
|       565|
+----------+
only showing top 5 rows



In [26]:
from pyspark.sql.functions import col, trim, count

df_imdb_votes.printSchema()

listas_vazias = df_imdb_votes.filter(
    (col("genres") == "[]") | (trim(col("genres")) == "")
)
print(f"Total de registos com 'genres' como '[]' ou vazio: {listas_vazias.count()}")

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

Total de registos com 'genres' como '[]' ou vazio: 58


In [27]:
from pyspark.sql.functions import regexp_replace, split, explode, lower

genres_exploded = df_imdb_votes \
    .filter( 
        (col("genres").isNotNull()) & 
        (col("genres") != "[]") &
        (trim(col("genres")) != "")
    ) \
    .withColumn(
        "genre_array",
        split(regexp_replace(col("genres"), "[\\['\\]]", ""), ", ") 
    ) \
    .select(explode(col("genre_array")).alias("genre")) \
    .select(trim(lower(col("genre"))).alias("genre")) \
    .distinct()

# Recolhe a lista (Ação .collect())
distinct_genres_list = [row['genre'] for row in genres_exploded.collect() if row['genre']]
num_distinct_genres = len(distinct_genres_list)

print(f"Encontrados {num_distinct_genres} géneros únicos.")

Encontrados 19 géneros únicos.


In [28]:
from pyspark.sql.functions import array_contains, transform, lit, when, col, trim, regexp_replace, split

df_temp_genres = df_imdb_votes.withColumn(
    "genres_array_cleaned",
    when(
        (col("genres").isNotNull()) & (col("genres") != "[]") & (trim(col("genres")) != ""),
        transform(
            split(regexp_replace(col("genres"), "[\\['\\]]", ""), ", "),
            lambda x: trim(lower(x))
        )
    ).otherwise(lit(None))
)

df_genres = df_temp_genres

for genre in distinct_genres_list:
    col_name = f"genre_{genre.replace('-', '_').replace(' ', '_')}" 
    df_genres = df_genres.withColumn(
        col_name,
        when(
            (col("genres_array_cleaned").isNotNull()) &
            (array_contains(col("genres_array_cleaned"), genre)),
            1
        ).otherwise(0)
    )

df_genres = df_genres.drop("genres", "genres_array_cleaned")

print(f"Transformação de 'genres' e criação de {len(distinct_genres_list)} colunas concluída.")

Transformação de 'genres' e criação de 19 colunas concluída.


In [29]:
print("Validando: Verificando o schema final após o tratamento de 'genres'...")

df_genres.printSchema()
df_genres.show(5)

Validando: Verificando o schema final após o tratamento de 'genres'...
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)
 |-- genre_scifi: integer (nullable = false)
 |-- genre_documentation: integer (nullable = false)
 |-- genre_crime: integer (nullable = false)
 |-- genre_fantasy: integer (nullable = false)
 |-- genre_action: integer (nullable = false)
 |-- genre_animation: integer (nullable = false)
 |-- genre_sport: integer (nullable = false)
 |-- 

In [30]:
from pyspark.sql.functions import col, trim, count
df_genres.printSchema() # Vai mostrar que é 'string'
listas_vazias_countries = df_genres.filter(
    (col("production_countries") == "[]") | (trim(col("production_countries")) == "")
)
print(f"Total de registos com 'production_countries' como '[]' ou vazio: {listas_vazias_countries.count()}")

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)
 |-- genre_scifi: integer (nullable = false)
 |-- genre_documentation: integer (nullable = false)
 |-- genre_crime: integer (nullable = false)
 |-- genre_fantasy: integer (nullable = false)
 |-- genre_action: integer (nullable = false)
 |-- genre_animation: integer (nullable = false)
 |-- genre_sport: integer (nullable = false)
 |-- genre_family: integer (nullable = false)
 |-- genre_horror: integer (nu

In [31]:
from pyspark.sql.functions import transform, lit, when, col, trim, regexp_replace, split

#Cria a nova coluna 'countries_array'
df_production_countries = df_genres.withColumn(
    "countries_array",
    when(
        (col("production_countries").isNotNull()) & 
        (col("production_countries") != "[]") & 
        (trim(col("production_countries")) != ""),
        transform(
            # Remove [ ] ' e divide
            split(regexp_replace(col("production_countries"), "[\\['\\]]", ""), ", "),
            # Limpa e padroniza cada item (ex: ' US ' -> 'us')
            lambda x: trim(lower(x)) 
        )
    ).otherwise(lit(None)) # Deixa nulo se era '[]' ou nulo
)

# Limpa a coluna original ('string') e renomeia a nova ('array')
df_production_countries = df_production_countries.drop("production_countries") \
                                             .withColumnRenamed("countries_array", "production_countries")


In [32]:
# Deve mostrar 'production_countries: array (nullable = true)'
df_production_countries.printSchema()
# Deve mostrar [us, gb], etc.
df_production_countries.select("id", "title", "production_countries").show(10, truncate=False)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- seasons: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)
 |-- genre_scifi: integer (nullable = false)
 |-- genre_documentation: integer (nullable = false)
 |-- genre_crime: integer (nullable = false)
 |-- genre_fantasy: integer (nullable = false)
 |-- genre_action: integer (nullable = false)
 |-- genre_animation: integer (nullable = false)
 |-- genre_sport: integer (nullable = false)
 |-- genre_family: integer (nullable = false)
 |-- genre_horror: integer (nullable = false)
 |-- genre_history: integer (nullabl

In [33]:
from pyspark.sql.functions import lower, trim, col, translate, regexp_replace

# Colunas de texto a limpar
text_columns_to_clean = ["title", "type", "description", "age_certification"]

# O nosso DataFrame de trabalho
df_final_limpo = df_production_countries

# Mapa de acentos
accent_map = {
    'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
    'ã': 'a', 'õ': 'o', 'â': 'a', 'ê': 'e', 'ô': 'o', 'ç': 'c',
    'ñ': 'n', 'ü': 'u', 'ë': 'e', 'ï': 'i', 'ö': 'o',
    'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
    'Ã': 'A', 'Õ': 'O', 'Â': 'A', 'Ê': 'E', 'Ô': 'O', 'Ç': 'C',
    'Ñ': 'N', 'Ü': 'U', 'Ë': 'E', 'Ï': 'I', 'Ö': 'O'
}

# Converte o mapa em duas strings para a função translate()
# Esta é a forma mais eficiente de o fazer em Spark
from_chars = "".join(accent_map.keys())
to_chars = "".join(accent_map.values())


for col_name in text_columns_to_clean:

# Começa com a coluna original
    cleaned_col = col(col_name)

    # Remove espaços no início/fim (Trim)
    cleaned_col = trim(cleaned_col)

    # Remove acentos (Diacritic Folding)
    # Usamos translate() que é muito mais rápido que múltiplos regexp_replace
    cleaned_col = translate(cleaned_col, from_chars, to_chars)

    # Converte para minúsculo (Lowercase)
    cleaned_col = lower(cleaned_col)

    # Remover espaços duplos
    cleaned_col = regexp_replace(cleaned_col, " +", " ")

    # Atualiza o DataFrame com a coluna limpa
    df_final_limpo = df_final_limpo.withColumn(col_name, cleaned_col)

# Mostra o resultado
df_final_limpo.select(text_columns_to_clean).show(truncate=False)

+----------------------------------------+-----+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [34]:
df_final_limpo \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("hdfs://hdfs-nn:9000/projeto/silver/netflix_titles")

In [35]:
df_final_limpo.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("release_year") \
    .option("overwriteSchema", "true") \
    .option("path", "hdfs://hdfs-nn:9000/warehouse/projeto.db/netflix_titles") \
    .saveAsTable("projeto.netflix_titles")

In [36]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.netflix_titles
    """
).show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2025-11-17 16:51:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|          0|  Serializable|        false|{numFiles -> 63, ...|        null|Apache-Spark/3.4....|
|      0|2025-11-16 23:41:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|       null|  Serializable|        false|{numFiles -

In [37]:
spark.sql(
    """
    SELECT * FROM projeto.netflix_titles
    """
).show()

+--------+--------------------+-----+--------------------+------------+-----------------+-------+-------+---------+----------+----------+---------------+----------+-----------+-------------------+-----------+-------------+------------+---------------+-----------+------------+------------+-------------+-----------+-----------+-------------+---------+--------------+-------------+--------------+-------------+------------+--------------------+
|      id|               title| type|         description|release_year|age_certification|runtime|seasons|  imdb_id|imdb_score|imdb_votes|tmdb_popularity|tmdb_score|genre_scifi|genre_documentation|genre_crime|genre_fantasy|genre_action|genre_animation|genre_sport|genre_family|genre_horror|genre_history|genre_music|genre_drama|genre_western|genre_war|genre_european|genre_romance|genre_thriller|genre_reality|genre_comedy|production_countries|
+--------+--------------------+-----+--------------------+------------+-----------------+-------+-------+-------

In [38]:
spark.stop()